In [13]:
import sys
sys.path.append('DA2_tools') # Make sure DA2_tools is accessible

import numpy as np
import h5py
import open3d as o3d
from DA2_tools import create_robotiq_marker
from scipy.spatial.transform import Rotation as R
from pathlib import Path
import time
import viser

# --- 1. Setup and Data Loading ---
server = viser.ViserServer()

def compute_camera_wrt_base(roll, pitch, yaw, x_mm, y_mm, z_mm):
    # This function is copied directly from your new script
    roll, pitch, yaw = np.deg2rad(roll), np.deg2rad(pitch), np.deg2rad(yaw)
    Rx = np.array([[1, 0, 0], [0, np.cos(roll), -np.sin(roll)], [0, np.sin(roll), np.cos(roll)]])
    Ry = np.array([[np.cos(pitch), 0, np.sin(pitch)], [0, 1, 0], [-np.sin(pitch), 0, np.cos(pitch)]])
    Rz = np.array([[np.cos(yaw), -np.sin(yaw), 0], [np.sin(yaw), np.cos(yaw), 0], [0, 0, 1]])
    R_base = Rz @ Ry @ Rx
    T_eef_wrt_base = np.eye(4)
    T_eef_wrt_base[:3, :3] = R_base
    T_eef_wrt_base[:3, 3] = [x_mm / 1000.0, y_mm / 1000.0, z_mm / 1000.0]
    R_cam = R.from_euler('xyz', [0, 0, -np.pi / 2], degrees=False).as_matrix()
    T_cam_wrt_eef = np.eye(4)
    T_cam_wrt_eef[:3, :3] = R_cam
    T_cam_wrt_eef[:3, 3] = [0.08, 0, 0.04]
    return T_eef_wrt_base @ T_cam_wrt_eef

T_cam_wrt_base = compute_camera_wrt_base(x_mm=-25.4, y_mm=328, z_mm=277.3, roll=-175.7, pitch=-62, yaw=-7.5)
theta = np.pi
rotation_matrix_z = np.array([[np.cos(theta), -np.sin(theta), 0, 0], [np.sin(theta), np.cos(theta), 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
T_cam_wrt_base = T_cam_wrt_base @ rotation_matrix_z

basepath = './blue_cylinder/experiment_dir/registered_meshes/'
# Load the object mesh WITH TEXTURE. We will not paint over it.
object_mesh = o3d.io.read_triangle_mesh(f'{basepath}0.obj', True) # Set True to enable post-processing
object_mesh.compute_vertex_normals()
object_mesh.transform(T_cam_wrt_base)

pcd = o3d.io.read_point_cloud(f'{basepath}scene_complete.ply')
pcd.transform(T_cam_wrt_base)

grasp_path = f"{basepath}0_decomposed_1.h5"
grasps_data = h5py.File(grasp_path, 'r')
grasps = grasps_data['grasps/transforms'][:].reshape(-1, 4, 4)
end_points = grasps_data['grasps/end_points'][:]
print(f"Loaded {len(grasps)} grasp transforms")

# --- NEW: Crop Point Cloud ---
print("Cropping point cloud where x < 0.5...")
points = np.asarray(pcd.points)
crop_indices = np.where(points[:, 0] < 0.6)[0]
cropped_pcd = pcd.select_by_index(crop_indices)
print(f"  - Original points: {len(pcd.points)}, Cropped points: {len(cropped_pcd.points)}")

# --- NEW: Plane Segmentation and Coloring on CROPPED cloud ---
print("Detecting plane on cropped point cloud...")
if not cropped_pcd.has_colors():
    cropped_pcd.paint_uniform_color([0.5, 0.5, 0.5])

plane_model, inliers = cropped_pcd.segment_plane(distance_threshold=0.005, ransac_n=3, num_iterations=1000)
print(f"  - Found {len(inliers)} points belonging to the plane.")
final_colors = np.asarray(cropped_pcd.colors)
final_colors[inliers] = [0.9, 0.8, 0.9] # Color the plane points

# --- Setup for Visualization ---
geometries_for_open3d = [object_mesh] # Start with the textured mesh
# Create a new point cloud object with the modified colors for the preview
final_pcd_for_preview = o3d.geometry.PointCloud()
final_pcd_for_preview.points = cropped_pcd.points
final_pcd_for_preview.colors = o3d.utility.Vector3dVector(final_colors)
geometries_for_open3d.append(final_pcd_for_preview)
geometries_for_open3d.append(o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1, origin=[0,0,0]))

# --- Add Static Scene Elements to Viser ---
server.add_point_cloud(name="/scene/point_cloud", points=np.asarray(cropped_pcd.points), colors=final_colors, point_size=0.002)
server.add_mesh(name="/scene/object_mesh", vertices=np.asarray(object_mesh.vertices), faces=np.asarray(object_mesh.triangles))

# --- Main Geometry Creation Loop ---
grasp_indices = [34, 180, 203, 61, 211, 259, 163, 116, 71, 136]
all_viser_handles = []

for idx in grasp_indices:
    # (The grasp transformation logic remains the same as before)
    grasp_camT = grasps[idx]
    grasp_worldT = T_cam_wrt_base @ grasp_camT
    gripper_trimesh = create_robotiq_marker()
    if hasattr(gripper_trimesh, 'dump'): gripper_trimesh = gripper_trimesh.dump().sum()
    gripper_o3d = o3d.geometry.TriangleMesh(o3d.utility.Vector3dVector(gripper_trimesh.vertices), o3d.utility.Vector3iVector(gripper_trimesh.faces))
    gripper_o3d.paint_uniform_color([0.1, 0.7, 0.1])
    gripper_o3d.transform(grasp_worldT)
    contact_spheres = []
    for point in end_points[idx]:
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.003).translate(point).transform(T_cam_wrt_base).paint_uniform_color([1, 0, 0])
        contact_spheres.append(sphere)
    R_mat_XG = grasp_worldT[:3, :3].copy()
    T_mat_XG = grasp_worldT[:3, 3].copy()
    R_mat_XG[:, [1, 2]] = R_mat_XG[:, [2, 1]]; R_mat_XG[:, 0] *= -1
    Rz_90 = np.array([[np.cos(-np.pi/2), -np.sin(-np.pi/2), 0], [np.sin(-np.pi/2), np.cos(-np.pi/2), 0], [0, 0, 1]])
    R_mat_XG = R_mat_XG @ Rz_90
    t_moved = T_mat_XG - 0.09 * R_mat_XG[:, 2]
    T_robot_final = np.eye(4)
    T_robot_final[:3, :3], T_robot_final[:3, 3] = R_mat_XG, t_moved
    robot_tcp_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.05).transform(T_robot_final)
    
    geometries_for_open3d.extend([gripper_o3d, robot_tcp_frame] + contact_spheres)
    current_grasp_viser_handles = []
    current_grasp_viser_handles.append(server.add_mesh(name=f"/grasps/grasp_{idx}/gripper", vertices=np.asarray(gripper_o3d.vertices), faces=np.asarray(gripper_o3d.triangles), color=(0.1, 0.7, 0.1), visible=False))
    # current_grasp_viser_handles.append(server.add_frame(name=f"/grasps/grasp_{idx}/tcp_frame", wxyz=R.from_matrix(T_robot_final[:3,:3]).as_quat()[[3,0,1,2]], position=T_robot_final[:3,3], axes_length=0.05, visible=False))
    for i, sphere in enumerate(contact_spheres):
        current_grasp_viser_handles.append(server.add_mesh(name=f"/grasps/grasp_{idx}/contact_{i}", vertices=np.asarray(sphere.vertices), faces=np.asarray(sphere.triangles), color=(1.0, 0.0, 0.0), visible=False))
    all_viser_handles.append(current_grasp_viser_handles)

# --- Dynamic Scene Export (Animation) FIRST ---
print("\nRecording Viser animation...")
recorder = server._start_scene_recording()
recorder.set_loop_start()

for i, handle_group in enumerate(all_viser_handles):
    print(f"  - Frame {i+1}/{len(all_viser_handles)}")
    for handle in handle_group: handle.visible = True
    recorder.insert_sleep(1.0)
    for handle in handle_group: handle.visible = False

# Save the Final Animation
output_filename = "grasp_animation_final2.viser"
print(f"Saving animation to {output_filename}...")
Path(output_filename).write_bytes(recorder.end_and_serialize())
print("File saved successfully.")

# --- Show the Open3D Preview Window LAST ---
print("\n.viser file saved. Now showing static preview in Open3D window.")
print("Close the Open3D window to exit the script.")
o3d.visualization.draw_geometries(
    geometries_for_open3d,
    window_name="Static Preview (after saving .viser)"
)

print("Script finished.")

╭──────────────── viser ────────────────╮
│             ╷                         │
│   HTTP      │ http://localhost:8098   │
│   Websocket │ ws://localhost:8098     │
│             ╵                         │
╰───────────────────────────────────────╯

Loaded 305 grasp transforms
Cropping point cloud where x < 0.5...
  - Original points: 254452, Cropped points: 85776
Detecting plane on cropped point cloud...
  - Found 80822 points belonging to the plane.

Recording Viser animation...
  - Frame 1/10
  - Frame 2/10
  - Frame 3/10
  - Frame 4/10
  - Frame 5/10
  - Frame 6/10
  - Frame 7/10
  - Frame 8/10
  - Frame 9/10
  - Frame 10/10
Saving animation to grasp_animation_final2.viser...
File saved successfully.

.viser file saved. Now showing static preview in Open3D window.
Close the Open3D window to exit the script.


/tmp/ipykernel_9460/3524531139.py:79: DeprecationWarning: ViserServer.add_point_cloud has been deprecated, use ViserServer.scene.add_point_cloud instead. Alternatively, pin to `viser<0.2.0`.
  server.add_point_cloud(name="/scene/point_cloud", points=np.asarray(cropped_pcd.points), colors=final_colors, point_size=0.002)
/tmp/ipykernel_9460/3524531139.py:80: DeprecationWarning: ViserServer.add_mesh has been deprecated, use ViserServer.scene.add_mesh_simple instead. Alternatively, pin to `viser<0.2.0`.
  server.add_mesh(name="/scene/object_mesh", vertices=np.asarray(object_mesh.vertices), faces=np.asarray(object_mesh.triangles))
/tmp/ipykernel_9460/3524531139.py:111: DeprecationWarning: ViserServer.add_mesh has been deprecated, use ViserServer.scene.add_mesh_simple instead. Alternatively, pin to `viser<0.2.0`.
  current_grasp_viser_handles.append(server.add_mesh(name=f"/grasps/grasp_{idx}/gripper", vertices=np.asarray(gripper_o3d.vertices), faces=np.asarray(gripper_o3d.triangles), color=(

Script finished.
